In [ ]:
!git clone https://github.com/zjunlp/EasyEdit.git
%cd EasyEdit

In [ ]:
!rm -rf /kaggle/working/pydeps
!mkdir -p /kaggle/working/pydeps

In [ ]:
!python -m pip install --no-cache-dir --upgrade --target /kaggle/working/pydeps \
  "numpy==1.26.4" \
  "scipy==1.13.1" \
  "scikit-learn==1.5.2" \
  "PyYAML==6.0.2" \
  "transformers==4.45.2" \
  "sentence-transformers==3.2.1" \
  "accelerate>=0.30.0" \
  "datasets" \
  "einops" \
  "higher" \
  "hydra-core" \
  "omegaconf" \
  "peft" \
  "tqdm" \
  "nltk" \
  "pandas"

In [19]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [20]:
import torch

print(torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

2
0 Tesla T4
1 Tesla T4


In [ ]:
import sys

PYDEPS = "/kaggle/working/pydeps"
REPO = "/kaggle/working/EasyEdit"

# Put our clean dependency dir first, before Kaggle's global site-packages
sys.path.insert(0, PYDEPS)
sys.path.insert(0, REPO)

print(sys.path[:5])

In [ ]:
import numpy, scipy, sklearn, transformers, torch

print("numpy:", numpy.__version__, numpy.__file__)
print("scipy:", scipy.__version__, scipy.__file__)
print("sklearn:", sklearn.__version__, sklearn.__file__)
print("transformers:", transformers.__version__, transformers.__file__)
print("torch:", torch.__version__, torch.__file__)
print("cuda:", torch.cuda.is_available())

In [ ]:
from pathlib import Path
import shutil

trainer_init = Path("/kaggle/working/EasyEdit/easyeditor/trainer/__init__.py")
backup = trainer_init.with_suffix(".py.bak2")
shutil.copy2(trainer_init, backup)

trainer_init.write_text("""from .training_hparams import *
from .EditTrainer import *
from .BaseTrainer import *

# expose training algorithms such as MEND
from .algs import *

# keep multimodal imports optional
try:
    from .blip2_models import *
    from .MultimodalTrainer import *
    from .MultiTaskTrainer import *
except ImportError:
    pass
""")

print(trainer_init.read_text())
print(f"Backup saved to: {backup}")

In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/kaggle/working/EasyEdit/easyeditor")

files_to_patch = {
    ROOT / "__init__.py": """# Minimal top-level imports for text-only editing
from .dataset import *
from .editors import *
from .util import *
""",
    ROOT / "dataset" / "__init__.py": """# Only the dataset needed for IKE here
from .zsre import ZsreDataset
""",
    ROOT / "editors" / "__init__.py": """# Only the standard text editor
from .editor import *
""",
    ROOT / "models" / "__init__.py": """# Only text-editing methods we want right now
from .ike import *
from .rome import *
""",
    ROOT / "trainer" / "__init__.py": """# Keep trainer minimal; not using MEND yet
from .training_hparams import *
from .EditTrainer import *
from .BaseTrainer import *
from .algs import *
""",
}

for path, new_text in files_to_patch.items():
    backup = path.with_suffix(path.suffix + ".bak")
    if not backup.exists():
        shutil.copy2(path, backup)
    path.write_text(new_text)
    print(f"Patched {path}")
    print(path.read_text())
    print("-" * 60)

In [ ]:
!python -m pip install --no-cache-dir --target /kaggle/working/pydeps rouge==1.0.1

In [ ]:
from pathlib import Path
import shutil

alg_dict_path = Path("/kaggle/working/EasyEdit/easyeditor/util/alg_dict.py")
backup = alg_dict_path.with_suffix(".py.bak")
if not backup.exists():
    shutil.copy2(alg_dict_path, backup)

alg_dict_path.write_text("""from ..models.ike import IKEHyperParams, apply_ike_to_model
from ..models.rome import ROMEHyperParams, apply_rome_to_model

ALG_DICT = {
    "IKE": apply_ike_to_model,
    "ROME": apply_rome_to_model,
}

ALG_MULTIMODAL_DICT = {}

HPARAMS_DICT = {
    "IKE": IKEHyperParams,
    "ROME": ROMEHyperParams,
}
""")

print(alg_dict_path.read_text())
print(f"Backup saved to: {backup}")

In [ ]:
!python -m pip uninstall -y torchvision

In [ ]:
!pip install -U "transformers @ git+https://github.com/huggingface/transformers.git@main" accelerate safetensors

In [ ]:
from pathlib import Path
import shutil

ROOT = Path("/kaggle/working/EasyEdit/easyeditor")

# 1. Patch trainer/__init__.py so importing easyeditor.trainer.utils
#    does NOT execute EditTrainer/BaseTrainer/model imports.
trainer_init = ROOT / "trainer" / "__init__.py"
backup = trainer_init.with_suffix(".py.bak_no_multimodal")

if not backup.exists():
    shutil.copy2(trainer_init, backup)

trainer_init.write_text("""# Minimal trainer package init for text-only IKE/ROME experiments.
# Do not import EditTrainer, BaseTrainer, algs, or trainer.models here.
# ZsreDataset only needs easyeditor.trainer.utils.dict_to.
""")

print("Patched:", trainer_init)
print(trainer_init.read_text())


# 2. Keep top-level easyeditor minimal.
top_init = ROOT / "__init__.py"
backup = top_init.with_suffix(".py.bak_text_only")

if not backup.exists():
    shutil.copy2(top_init, backup)

top_init.write_text("""# Minimal top-level imports for text-only IKE/ROME experiments.
from .dataset import *
from .editors import *
from .util import *
""")

print("Patched:", top_init)
print(top_init.read_text())


# 3. Keep dataset init minimal.
dataset_init = ROOT / "dataset" / "__init__.py"
backup = dataset_init.with_suffix(".py.bak_text_only")

if not backup.exists():
    shutil.copy2(dataset_init, backup)

dataset_init.write_text("""# Only the dataset needed for IKE/ZsRE.
from .zsre import ZsreDataset
""")

print("Patched:", dataset_init)
print(dataset_init.read_text())


# 4. Keep models init text-only.
models_init = ROOT / "models" / "__init__.py"
backup = models_init.with_suffix(".py.bak_text_only")

if not backup.exists():
    shutil.copy2(models_init, backup)

models_init.write_text("""# Only text-editing methods needed now.
from .ike import *
from .rome import *
""")

print("Patched:", models_init)
print(models_init.read_text())

In [ ]:
from pathlib import Path

editor_path = Path("/kaggle/working/EasyEdit/easyeditor/editors/editor.py")

text = editor_path.read_text()

old = "self.model = AutoModelForCausalLM.from_pretrained(self.model_name,fp32=False,trust_remote_code=True, **model_kwargs)"
new = """self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_name,
                    trust_remote_code=True,
                    **model_kwargs
                )"""

if old not in text:
    print("Original line not found. It may already be patched or formatted differently.")
else:
    text = text.replace(old, new)
    editor_path.write_text(text)
    print("Patched EasyEdit editor.py: removed fp32=False for Qwen.")

In [3]:
import sys

PYDEPS = "/kaggle/working/pydeps"
REPO = "/kaggle/working/EasyEdit"

for p in [PYDEPS, REPO]:
    if p not in sys.path:
        sys.path.insert(0, p)

from easyeditor.editors.editor import BaseEditor
from easyeditor.models.ike.ike_hparams import IKEHyperParams
from easyeditor.models.rome.rome_hparams import ROMEHyperParams
from easyeditor.dataset.zsre import ZsreDataset

print("Targeted imports OK")

Targeted imports OK


In [4]:
import os
import sys
import json
import pickle
import pprint
import re
import shutil
from pathlib import Path

import torch
import pandas as pd

PYDEPS = "/kaggle/working/pydeps"
REPO = "/kaggle/working/EasyEdit"

for p in [PYDEPS, REPO]:
    if p not in sys.path:
        sys.path.insert(0, p)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download
from sentence_transformers import SentenceTransformer, util

print("Imports OK")

Imports OK


In [5]:
hf_token = UserSecretsClient().get_secret("HF_TOKEN").strip()

os.environ["hf_token"] = hf_token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

login(token=hf_token)

print("HF login OK")

05/03/2026 23:22:28 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"


HF login OK


In [6]:
DATA_DIR = Path("/kaggle/working/EasyEdit/data/zsre_real")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "zsre_mend_train_10000.json"
EVAL_PATH = DATA_DIR / "zsre_mend_eval_portability_gpt4.json"

train_src = hf_hub_download(
    repo_id="wangzn2001/data",
    repo_type="dataset",
    filename="data/zsre/zsre_mend_train_10000.json",
)

eval_src = hf_hub_download(
    repo_id="wangzn2001/data",
    repo_type="dataset",
    filename="data/portability/One Hop/zsre_mend_eval_portability_gpt4.json",
)

shutil.copy(train_src, TRAIN_PATH)
shutil.copy(eval_src, EVAL_PATH)

print("Downloaded train:", TRAIN_PATH)
print("Downloaded eval:", EVAL_PATH)
print("Train size MB:", TRAIN_PATH.stat().st_size / 1e6)
print("Eval size MB:", EVAL_PATH.stat().st_size / 1e6)

05/03/2026 23:22:28 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/datasets/wangzn2001/data/resolve/main/data/zsre/zsre_mend_train_10000.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:22:28 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/wangzn2001/data/0f3666b7646e47a44c636667b3aa6e18a04ea33b/data%2Fzsre%2Fzsre_mend_train_10000.json "HTTP/1.1 200 OK"
05/03/2026 23:22:28 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/datasets/wangzn2001/data/resolve/main/data/portability/One%20Hop/zsre_mend_eval_portability_gpt4.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:22:28 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/wangzn2001/data/0f3666b7646e47a44c636667b3aa6e18a04ea33b/data%2Fportability%2FOne%20Hop%2Fzsre_mend_eval_portability_gpt4.json "HTTP/1.1 200 OK"


Downloaded train: /kaggle/working/EasyEdit/data/zsre_real/zsre_mend_train_10000.json
Downloaded eval: /kaggle/working/EasyEdit/data/zsre_real/zsre_mend_eval_portability_gpt4.json
Train size MB: 5.277557
Eval size MB: 0.842861


In [7]:
train_raw = json.load(open(TRAIN_PATH, "r", encoding="utf-8"))
eval_raw = json.load(open(EVAL_PATH, "r", encoding="utf-8"))

print("Raw train rows:", len(train_raw))
print("Raw eval rows:", len(eval_raw))

print("\nTrain example:")
print(json.dumps(train_raw[0], indent=2, ensure_ascii=False)[:2500])

print("\nEval example:")
print(json.dumps(eval_raw[0], indent=2, ensure_ascii=False)[:2500])

Raw train rows: 10000
Raw eval rows: 1037

Train example:
{
  "subject": "Christiane Cohendy",
  "src": "What is the native language of Christiane Cohendy?",
  "pred": "French",
  "rephrase": "What's Christiane Cohendy's mother tongue?",
  "alt": "German",
  "answers": [
    "French"
  ],
  "loc": "nq question: what is the most current season of the walking dead",
  "loc_ans": "The eighth season",
  "cond": "French >> German || What is the native language of Christiane Cohendy?"
}

Eval example:
{
  "subject": "IAAF Combined Events Challenge",
  "src": "When was the inception of IAAF Combined Events Challenge?",
  "pred": "2011",
  "rephrase": "When was the IAAF Combined Events Challenge launched?",
  "alt": "2006",
  "answers": [
    "1998"
  ],
  "loc": "nq question: what is the name of the last episode of spongebob",
  "loc_ans": "The String",
  "cond": "2011 >> 2006 || When was the inception of IAAF Combined Events Challenge?",
  "portability": {
    "Recalled Relation": "(IAAF Com

In [8]:
TRAIN_SIZE = 1000
EVAL_SIZE = 5

train_ds = ZsreDataset(str(TRAIN_PATH), size=TRAIN_SIZE)

eval_all = json.load(open(EVAL_PATH, "r", encoding="utf-8"))

valid_test_data = []
for x in eval_all:
    required = ["src", "alt", "answers", "rephrase", "loc", "loc_ans", "subject"]
    if not all(k in x for k in required):
        continue
    if not x.get("alt"):
        continue
    if not x.get("answers"):
        continue
    if "portability" not in x:
        continue
    if "New Question" not in x["portability"]:
        continue
    if "New Answer" not in x["portability"]:
        continue
    valid_test_data.append(x)

test_data = valid_test_data[:EVAL_SIZE]

print("Loaded train_ds:", len(train_ds))
print("Valid eval rows:", len(valid_test_data))
print("Using eval rows:", len(test_data))

print("\nFirst train_ds item:")
pprint.pp(train_ds[0])

print("\nFirst eval item:")
print(json.dumps(test_data[0], indent=2, ensure_ascii=False)[:2500])

Loaded train_ds: 1000
Valid eval rows: 1037
Using eval rows: 5

First train_ds item:
{'case_id': 0,
 'prompt': 'What is the native language of Christiane Cohendy?',
 'target_new': 'German',
 'ground_truth': 'French',
 'rephrase_prompt': "What's Christiane Cohendy's mother tongue?",
 'locality_prompt': 'nq question: what is the most current season of the '
                    'walking dead',
 'locality_ground_truth': 'The eighth season',
 'cond': 'French >> German || What is the native language of Christiane '
         'Cohendy?'}

First eval item:
{
  "subject": "IAAF Combined Events Challenge",
  "src": "When was the inception of IAAF Combined Events Challenge?",
  "pred": "2011",
  "rephrase": "When was the IAAF Combined Events Challenge launched?",
  "alt": "2006",
  "answers": [
    "1998"
  ],
  "loc": "nq question: what is the name of the last episode of spongebob",
  "loc_ans": "The String",
  "cond": "2011 >> 2006 || When was the inception of IAAF Combined Events Challenge?",
 

In [9]:
HPARAMS_PATH = Path("/kaggle/working/EasyEdit/hparams/IKE/qwen3.5-4b.yaml")
HPARAMS_PATH.parent.mkdir(parents=True, exist_ok=True)

HPARAMS_PATH.write_text("""
alg_name: "IKE"
model_name: "Qwen/Qwen3.5-4B"
sentence_model_name: "sentence-transformers/all-MiniLM-L6-v2"
device: 0
results_dir: "./results"
k: 4
model_parallel: false
""".strip())

print(HPARAMS_PATH.read_text())

alg_name: "IKE"
model_name: "Qwen/Qwen3.5-4B"
sentence_model_name: "sentence-transformers/all-MiniLM-L6-v2"
device: 0
results_dir: "./results"
k: 4
model_parallel: false


In [10]:
from easyeditor.models.ike.ike_hparams import IKEHyperParams
from easyeditor.models.ike import encode_ike_facts
from sentence_transformers import SentenceTransformer

hparams = IKEHyperParams.from_hparams(str(HPARAMS_PATH))

sentence_model = SentenceTransformer(hparams.sentence_model_name).to(f"cuda:{hparams.device}")

encode_ike_facts(sentence_model, train_ds, hparams)

print("IKE facts encoded")

05/03/2026 23:22:29 - INFO - sentence_transformers.SentenceTransformer -   Use pytorch device_name: cuda:0
05/03/2026 23:22:29 - INFO - sentence_transformers.SentenceTransformer -   Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
05/03/2026 23:22:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:22:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
05/03/2026 23:22:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:22:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

05/03/2026 23:22:30 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:22:30 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
05/03/2026 23:22:30 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:22:30 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
05/03/2026 23:22:30 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=f

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

IKE facts encoded


In [11]:
prompts = [x["src"] for x in test_data]
rephrase_prompts = [x["rephrase"] for x in test_data]
target_new = [x["alt"] for x in test_data]
ground_truth = [x["answers"][0] for x in test_data]
subject = [x["subject"] for x in test_data]

locality_inputs = {
    "neighborhood": {
        "prompt": [x["loc"] for x in test_data],
        "ground_truth": [x["loc_ans"] for x in test_data],
    }
}

portability_inputs = {
    "one_hop": {
        "prompt": [x["portability"]["New Question"] for x in test_data],
        "ground_truth": [x["portability"]["New Answer"] for x in test_data],
    }
}

print("Example prompt:", prompts[0])
print("Example target_new:", target_new[0])
print("Example ground_truth:", ground_truth[0])
print("Example portability prompt:", portability_inputs["one_hop"]["prompt"][0])
print("Example portability answer:", portability_inputs["one_hop"]["ground_truth"][0])

Example prompt: When was the inception of IAAF Combined Events Challenge?
Example target_new: 2006
Example ground_truth: 1998
Example portability prompt: What type of sports event is the IAAF Combined Events Challenge, which was established in 2006?
Example portability answer: Athletics


In [12]:
editor = BaseEditor.from_hparams(hparams)

metrics, edited_model, _ = editor.edit(
    prompts=prompts,
    rephrase_prompts=rephrase_prompts,
    target_new=target_new,
    ground_truth=ground_truth,
    subject=subject,
    train_ds=train_ds,
    locality_inputs=locality_inputs,
    portability_inputs=portability_inputs,
    keep_original_weight=True,
)

print("EasyEdit strict metrics:")
pprint.pp(metrics)

2026-05-03 23:22:33,699 - easyeditor.editors.editor - INFO - Instantiating model
05/03/2026 23:22:33 - INFO - easyeditor.editors.editor -   Instantiating model
05/03/2026 23:22:33 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:22:33 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
05/03/2026 23:22:33 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:22:33 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
05/03/2026 23:22:33 - INFO - httpx -   HTTP Request: HEAD https://huggingface

model.safetensors.index.json: 0.00B [00:00, ?B/s]

05/03/2026 23:22:34 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-4B/revision/main "HTTP/1.1 200 OK"


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

05/03/2026 23:22:34 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/model.safetensors-00001-of-00002.safetensors "HTTP/1.1 302 Found"
05/03/2026 23:22:34 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/model.safetensors-00002-of-00002.safetensors "HTTP/1.1 302 Found"
05/03/2026 23:22:34 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-4B/xet-read-token/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a "HTTP/1.1 200 OK"
05/03/2026 23:22:34 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-4B/xet-read-token/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a "HTTP/1.1 200 OK"
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installa

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

05/03/2026 23:25:24 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:24 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
05/03/2026 23:25:24 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:24 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
05/03/2026 23:25:24 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=f

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

05/03/2026 23:25:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
05/03/2026 23:25:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:29 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
05/03/2026 23:25:29 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=f

Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.1800000011920929), 'rephrase_acc': np.float64(0.7200000047683716), 'locality': {'neighborhood_acc': np.float64(0.76)}, 'portability': {'one_hop_acc': np.float64(0.44666666984558107)}}, 'post': {'rewrite_acc': np.float64(1.0), 'rephrase_acc': np.float64(1.0), 'locality': {'neighborhood_acc': np.float64(0.76)}, 'portability': {'one_hop_acc': np.float64(0.5933333396911621)}}}
EasyEdit strict metrics:
[{'pre': {'rewrite_acc': [0.20000000298023224],
          'locality': {'neighborhood_acc': np.float64(0.5)},
          'portability': {'one_hop_acc': 0.0},
          'rephrase_acc': 0.800000011920929},
  'case_id': 0,
  'requested_rewrite': {'prompt': 'When was the inception of IAAF Combined '
                                  'Events Challenge?',
                        'target_new': '2006',
                        'ground_truth': '1998',
                        'portability': {'one_hop': {'prompt': 'What type of '
                      

In [13]:
def normalize_answer(text):
    if text is None:
        return ""

    text = str(text).strip()

    text = re.sub(r"^(answer\s*:?\s*)+", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^(final answer\s*:?\s*)+", "", text, flags=re.IGNORECASE).strip()

    text = text.split("\n")[0].strip()
    text = re.split(r"\bExplanation\s*:", text, flags=re.IGNORECASE)[0].strip()
    text = text.strip(" .,:;!?\"'`")
    text = re.sub(r"\s+", " ", text.lower()).strip()

    return text


def relaxed_match(pred, gold):
    pred_norm = normalize_answer(pred)
    gold_norm = normalize_answer(gold)

    if not gold_norm:
        return False

    return (
        pred_norm == gold_norm
        or pred_norm.startswith(gold_norm)
        or gold_norm in pred_norm
    )


def get_model_and_tokenizer(editor, edited_model=None):
    tok = editor.tok if hasattr(editor, "tok") else editor.tokenizer
    model = edited_model if edited_model is not None else editor.model

    if tok.pad_token_id is None:
        tok.pad_token_id = tok.eos_token_id

    return model, tok


def load_ike_cache(hparams, train_ds, device):
    safe_model_name = hparams.sentence_model_name.rsplit("/", 1)[-1]
    emb_path = (
        Path(hparams.results_dir)
        / hparams.alg_name
        / "embedding"
        / f"{safe_model_name}_{type(train_ds).__name__}_{len(train_ds)}.pkl"
    )

    if not emb_path.exists():
        raise FileNotFoundError(
            f"IKE embedding cache not found: {emb_path}\n"
            "Run encode_ike_facts(sentence_model, train_ds, hparams) first."
        )

    with open(emb_path, "rb") as f:
        stored_data = pickle.load(f)

    stored_sentences = stored_data["sentences"]
    stored_embeddings = torch.tensor(stored_data["embeddings"]).to(device)
    stored_embeddings = util.normalize_embeddings(stored_embeddings)

    return stored_sentences, stored_embeddings, emb_path


def retrieve_ike_examples(
    sentence_model,
    stored_sentences,
    stored_embeddings,
    prompt,
    target_new,
    hparams,
    device,
):
    new_fact = prompt + " " + target_new
    query_sentence = f"New Fact: {new_fact}\nPrompt: {prompt}\n\n"

    query_embedding = sentence_model.encode(query_sentence, show_progress_bar=False)
    query_embedding = torch.tensor(query_embedding).unsqueeze(0).to(device)
    query_embedding = util.normalize_embeddings(query_embedding)

    hits = util.semantic_search(
        query_embedding,
        stored_embeddings,
        score_function=util.dot_score,
        top_k=hparams.k,
    )[0]

    icl_examples = [stored_sentences[h["corpus_id"]] for h in hits]
    return icl_examples, hits, query_sentence


def build_ike_prompt(icl_examples, edit_prompt, target_new, eval_prompt):
    x = f"New Fact: {edit_prompt} {target_new}\nPrompt: {eval_prompt}"
    return "".join(icl_examples) + x


def generate_continuation(model, tok, prompt, max_new_tokens=32):
    model.eval()
    device = next(model.parameters()).device

    inputs = tok(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )

    generated_ids = out[0][input_len:]
    continuation = tok.decode(generated_ids, skip_special_tokens=True)
    full = tok.decode(out[0], skip_special_tokens=True)

    return full, continuation


def robust_ike_eval(
    editor,
    edited_model,
    hparams,
    train_ds,
    test_data,
    max_new_tokens=32,
    show_prompts=False,
):
    device = torch.device(f"cuda:{hparams.device}")
    model, tok = get_model_and_tokenizer(editor, edited_model)

    sentence_model = SentenceTransformer(hparams.sentence_model_name).to(device)
    stored_sentences, stored_embeddings, emb_path = load_ike_cache(
        hparams, train_ds, device
    )

    print("Using IKE cache:", emb_path)
    print("Stored IKE examples:", len(stored_sentences))

    rows = []

    for case_id, x in enumerate(test_data):
        edit_prompt = x["src"]
        target = x["alt"]
        old_answer = x["answers"][0]

        icl_examples, hits, query_sentence = retrieve_ike_examples(
            sentence_model=sentence_model,
            stored_sentences=stored_sentences,
            stored_embeddings=stored_embeddings,
            prompt=edit_prompt,
            target_new=target,
            hparams=hparams,
            device=device,
        )

        eval_specs = [
            {
                "eval_type": "rewrite",
                "prompt": x["src"],
                "expected_post": x["alt"],
                "expected_pre": old_answer,
            },
            {
                "eval_type": "rephrase",
                "prompt": x["rephrase"],
                "expected_post": x["alt"],
                "expected_pre": old_answer,
            },
            {
                "eval_type": "locality",
                "prompt": x["loc"],
                "expected_post": x["loc_ans"],
                "expected_pre": x["loc_ans"],
            },
            {
                "eval_type": "portability",
                "prompt": x["portability"]["New Question"],
                "expected_post": x["portability"]["New Answer"],
                "expected_pre": None,
            },
        ]

        for spec in eval_specs:
            ike_prompt = build_ike_prompt(
                icl_examples=icl_examples,
                edit_prompt=edit_prompt,
                target_new=target,
                eval_prompt=spec["prompt"],
            )

            full, continuation = generate_continuation(
                model=model,
                tok=tok,
                prompt=ike_prompt,
                max_new_tokens=max_new_tokens,
            )

            pred_clean = normalize_answer(continuation)
            is_correct = relaxed_match(continuation, spec["expected_post"])

            row = {
                "case_id": case_id,
                "eval_type": spec["eval_type"],
                "subject": x.get("subject"),
                "eval_prompt": spec["prompt"],
                "expected_post": spec["expected_post"],
                "expected_pre": spec["expected_pre"],
                "raw_continuation": continuation,
                "normalized_prediction": pred_clean,
                "correct_relaxed": is_correct,
                "retrieved_ids": [h["corpus_id"] for h in hits],
                "retrieved_scores": [float(h["score"]) for h in hits],
            }

            if show_prompts:
                row["ike_prompt"] = ike_prompt
                row["retrieved_examples"] = icl_examples
                row["query_sentence"] = query_sentence

            rows.append(row)

    df = pd.DataFrame(rows)

    summary = (
        df.groupby("eval_type")["correct_relaxed"]
        .mean()
        .to_dict()
    )

    return df, summary

STOPWORDS = {
    "a", "an", "the", "of", "in", "on", "at", "to", "for", "from", "by", "with",
    "and", "or", "is", "are", "was", "were", "be", "been", "being", "as", "that",
    "which", "who", "what", "where", "when", "why", "how", "it", "its", "their",
    "his", "her", "he", "she", "they", "this", "these", "those", "there", "here",
    "answer", "prompt", "question", "new", "fact"
}

def normalize_answer_v2(text):
    if text is None:
        return ""

    text = str(text).strip()

    # Remove prompt residue / labels.
    text = re.sub(r"^(answer\s*:?\s*)+", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^(final answer\s*:?\s*)+", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^(prompt\s*:?\s*)+", "", text, flags=re.IGNORECASE).strip()

    # Stop at obvious demo continuation markers.
    text = re.split(r"\n\s*New Fact\s*:", text, flags=re.IGNORECASE)[0]
    text = re.split(r"\n\s*Prompt\s*:", text, flags=re.IGNORECASE)[0]
    text = re.split(r"\n\s*nq question\s*:", text, flags=re.IGNORECASE)[0]
    text = re.split(r"\bExplanation\s*:", text, flags=re.IGNORECASE)[0]

    # First line is usually the answer.
    text = text.split("\n")[0].strip()

    # Remove bracketed choices or True/False suffixes.
    text = re.sub(r"\b(True|False)\b.*$", "", text, flags=re.IGNORECASE).strip()

    # Light cleanup.
    text = text.strip(" .,:;!?\"'`()[]{}")
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize_answer(text):
    text = normalize_answer_v2(text)
    toks = re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text)
    toks = [t for t in toks if t not in STOPWORDS]
    return toks


def token_f1(pred, gold):
    pred_toks = tokenize_answer(pred)
    gold_toks = tokenize_answer(gold)

    if not pred_toks or not gold_toks:
        return 0.0

    pred_counts = {}
    gold_counts = {}

    for t in pred_toks:
        pred_counts[t] = pred_counts.get(t, 0) + 1
    for t in gold_toks:
        gold_counts[t] = gold_counts.get(t, 0) + 1

    overlap = 0
    for t in pred_counts:
        overlap += min(pred_counts[t], gold_counts.get(t, 0))

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_toks)
    recall = overlap / len(gold_toks)

    return 2 * precision * recall / (precision + recall)


def exact_or_contains(pred, gold):
    p = normalize_answer_v2(pred)
    g = normalize_answer_v2(gold)

    if not p or not g:
        return False

    return p == g or p.startswith(g) or g in p


def semantic_similarity(sentence_model, pred, gold, device):
    p = normalize_answer_v2(pred)
    g = normalize_answer_v2(gold)

    if not p or not g:
        return 0.0

    emb = sentence_model.encode([p, g], convert_to_tensor=True, show_progress_bar=False).to(device)
    emb = util.normalize_embeddings(emb)
    return float(util.dot_score(emb[0], emb[1]).item())


def robust_answer_score(
    pred,
    gold,
    sentence_model=None,
    device=None,
    f1_threshold=0.50,
    semantic_threshold=0.72,
):
    exact = exact_or_contains(pred, gold)
    f1 = token_f1(pred, gold)

    sem = None
    if sentence_model is not None and device is not None:
        sem = semantic_similarity(sentence_model, pred, gold, device)

    correct = exact or f1 >= f1_threshold or (sem is not None and sem >= semantic_threshold)

    return {
        "normalized_prediction_v2": normalize_answer_v2(pred),
        "normalized_gold_v2": normalize_answer_v2(gold),
        "exact_or_contains": exact,
        "token_f1": f1,
        "semantic_similarity": sem,
        "correct_relaxed_v2": correct,
    }


In [14]:
import os
import gc
import json
import time
import pprint
from pathlib import Path

import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

from easyeditor.dataset.zsre import ZsreDataset
from easyeditor.editors.editor import BaseEditor
from easyeditor.models.ike.ike_hparams import IKEHyperParams
from easyeditor.models.ike import encode_ike_facts

ROOT = Path("/kaggle/working/EasyEdit")
DATA_DIR = ROOT / "data" / "zsre_real"
OUT_DIR = ROOT / "output" / "ike_sweeps"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "zsre_mend_train_10000.json"
EVAL_PATH = DATA_DIR / "zsre_mend_eval_portability_gpt4.json"

assert TRAIN_PATH.exists(), TRAIN_PATH
assert EVAL_PATH.exists(), EVAL_PATH

eval_all = json.load(open(EVAL_PATH, "r", encoding="utf-8"))

valid_test_data = []
for x in eval_all:
    required = ["src", "alt", "answers", "rephrase", "loc", "loc_ans", "subject"]
    if not all(k in x for k in required):
        continue
    if not x.get("alt"):
        continue
    if not x.get("answers"):
        continue
    if "portability" not in x:
        continue
    if "New Question" not in x["portability"]:
        continue
    if "New Answer" not in x["portability"]:
        continue
    valid_test_data.append(x)

print("Valid eval rows:", len(valid_test_data))

HPARAMS_PATH = ROOT / "hparams" / "IKE" / "qwen3.5-4b.yaml"
HPARAMS_PATH.parent.mkdir(parents=True, exist_ok=True)

def write_ike_hparams(k):
    HPARAMS_PATH.write_text(f"""
alg_name: "IKE"
model_name: "Qwen/Qwen3.5-4B"
sentence_model_name: "sentence-transformers/all-MiniLM-L6-v2"
device: 0
results_dir: "./results"
k: {k}
model_parallel: false
""".strip())
    return IKEHyperParams.from_hparams(str(HPARAMS_PATH))

def to_float(x):
    if isinstance(x, (float, int)):
        return float(x)
    if isinstance(x, np.generic):
        return float(x)
    if isinstance(x, list):
        if len(x) == 0:
            return np.nan
        return float(np.mean([to_float(v) for v in x]))
    return float(x)


def extract_easyedit_summary(metrics):
    rows = []

    for case in metrics:
        row = {
            "case_id": case["case_id"],

            "ike_pre_rewrite": to_float(case["pre"]["rewrite_acc"]),
            "ike_pre_rephrase": to_float(case["pre"]["rephrase_acc"]),
            "ike_pre_locality": to_float(case["pre"]["locality"]["neighborhood_acc"]),
            "ike_pre_portability": to_float(case["pre"]["portability"]["one_hop_acc"]),

            "ike_post_rewrite": to_float(case["post"]["rewrite_acc"]),
            "ike_post_rephrase": to_float(case["post"]["rephrase_acc"]),
            "ike_post_locality": to_float(case["post"]["locality"]["neighborhood_acc"]),
            "ike_post_portability": to_float(case["post"]["portability"]["one_hop_acc"]),
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    return {
        "ike_pre_rewrite": df["ike_pre_rewrite"].mean(),
        "ike_pre_rephrase": df["ike_pre_rephrase"].mean(),
        "ike_pre_locality": df["ike_pre_locality"].mean(),
        "ike_pre_portability": df["ike_pre_portability"].mean(),

        "ike_post_rewrite": df["ike_post_rewrite"].mean(),
        "ike_post_rephrase": df["ike_post_rephrase"].mean(),
        "ike_post_locality": df["ike_post_locality"].mean(),
        "ike_post_portability": df["ike_post_portability"].mean(),
    }, df


def build_eval_inputs(test_data):
    prompts = [x["src"] for x in test_data]
    rephrase_prompts = [x["rephrase"] for x in test_data]
    target_new = [x["alt"] for x in test_data]
    ground_truth = [x["answers"][0] for x in test_data]
    subject = [x["subject"] for x in test_data]

    locality_inputs = {
        "neighborhood": {
            "prompt": [x["loc"] for x in test_data],
            "ground_truth": [x["loc_ans"] for x in test_data],
        }
    }

    portability_inputs = {
        "one_hop": {
            "prompt": [x["portability"]["New Question"] for x in test_data],
            "ground_truth": [x["portability"]["New Answer"] for x in test_data],
        }
    }

    return prompts, rephrase_prompts, target_new, ground_truth, subject, locality_inputs, portability_inputs


def rescore_robust_v2(robust_df, hparams):
    device = torch.device(f"cuda:{hparams.device}")
    eval_sentence_model = SentenceTransformer(hparams.sentence_model_name).to(device)

    rescored_rows = []

    for _, row in robust_df.iterrows():
        score = robust_answer_score(
            pred=row["raw_continuation"],
            gold=row["expected_post"],
            sentence_model=eval_sentence_model,
            device=device,
            f1_threshold=0.50,
            semantic_threshold=0.72,
        )

        out = row.to_dict()
        out.update(score)
        rescored_rows.append(out)

    robust_df_v2 = pd.DataFrame(rescored_rows)

    robust_summary_v2 = (
        robust_df_v2.groupby("eval_type")["correct_relaxed_v2"]
        .mean()
        .to_dict()
    )

    # Clean up sentence model memory.
    del eval_sentence_model
    torch.cuda.empty_cache()
    gc.collect()

    return robust_df_v2, robust_summary_v2

Valid eval rows: 1037


In [15]:
TRAIN_DS_CACHE = {}

def get_train_ds(train_size):
    if train_size not in TRAIN_DS_CACHE:
        print(f"Loading train_ds size={train_size}")
        TRAIN_DS_CACHE[train_size] = ZsreDataset(str(TRAIN_PATH), size=train_size)
    return TRAIN_DS_CACHE[train_size]


def run_one_ike_experiment(
    train_size,
    eval_size,
    k,
    run_name,
    editor=None,
    max_new_tokens=32,
):
    print("\n" + "=" * 100)
    print(f"RUN: {run_name}")
    print(f"TRAIN_SIZE={train_size}, EVAL_SIZE={eval_size}, k={k}")
    print("=" * 100)

    run_dir = OUT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    hparams = write_ike_hparams(k)
    train_ds = get_train_ds(train_size)
    test_data = valid_test_data[:eval_size]

    assert len(test_data) == eval_size, f"Only got {len(test_data)} eval rows"

    # Encode IKE facts. Cache path depends on train_ds length, not k.
    sentence_model = SentenceTransformer(hparams.sentence_model_name).to(f"cuda:{hparams.device}")
    encode_ike_facts(sentence_model, train_ds, hparams)
    del sentence_model
    torch.cuda.empty_cache()
    gc.collect()

    if editor is None:
        editor = BaseEditor.from_hparams(hparams)
    else:
        # Make sure editor uses current hparams/k.
        editor.hparams = hparams

    (
        prompts,
        rephrase_prompts,
        target_new,
        ground_truth,
        subject,
        locality_inputs,
        portability_inputs,
    ) = build_eval_inputs(test_data)

    print("First eval prompt:", prompts[0])
    print("First target_new:", target_new[0])
    print("First portability answer:", portability_inputs["one_hop"]["ground_truth"][0])

    start = time.time()

    metrics, edited_model, _ = editor.edit(
        prompts=prompts,
        rephrase_prompts=rephrase_prompts,
        target_new=target_new,
        ground_truth=ground_truth,
        subject=subject,
        train_ds=train_ds,
        locality_inputs=locality_inputs,
        portability_inputs=portability_inputs,
        keep_original_weight=True,
    )

    easy_summary, easy_case_df = extract_easyedit_summary(metrics)

    print("\nEasyEdit strict summary:")
    pprint.pp(easy_summary)

    robust_df, robust_summary = robust_ike_eval(
        editor=editor,
        edited_model=edited_model,
        hparams=hparams,
        train_ds=train_ds,
        test_data=test_data,
        max_new_tokens=max_new_tokens,
        show_prompts=False,
    )

    robust_df_v2, robust_summary_v2 = rescore_robust_v2(robust_df, hparams)

    print("\nRobust v1 summary:")
    pprint.pp(robust_summary)

    print("\nRobust v2 summary:")
    pprint.pp(robust_summary_v2)

    elapsed_sec = time.time() - start

    summary_row = {
        "run_name": run_name,
        "train_size": train_size,
        "eval_size": eval_size,
        "k": k,
        "elapsed_sec": elapsed_sec,

        **easy_summary,

        "robust_v1_rewrite": robust_summary.get("rewrite", np.nan),
        "robust_v1_rephrase": robust_summary.get("rephrase", np.nan),
        "robust_v1_locality": robust_summary.get("locality", np.nan),
        "robust_v1_portability": robust_summary.get("portability", np.nan),

        "robust_v2_rewrite": robust_summary_v2.get("rewrite", np.nan),
        "robust_v2_rephrase": robust_summary_v2.get("rephrase", np.nan),
        "robust_v2_locality": robust_summary_v2.get("locality", np.nan),
        "robust_v2_portability": robust_summary_v2.get("portability", np.nan),
    }

    # Save detailed outputs.
    with open(run_dir / "easyedit_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2, default=str)

    easy_case_df.to_csv(run_dir / "easyedit_case_metrics.csv", index=False)
    robust_df.to_csv(run_dir / "robust_v1_details.csv", index=False)
    robust_df_v2.to_csv(run_dir / "robust_v2_details.csv", index=False)

    with open(run_dir / "summary.json", "w") as f:
        json.dump(summary_row, f, indent=2, default=str)

    print(f"\nSaved run outputs to: {run_dir}")

    return summary_row, easy_case_df, robust_df, robust_df_v2, editor

In [16]:
base_hparams = write_ike_hparams(k=4)

base_hparams.model_parallel = True
base_hparams.device = 0

editor = BaseEditor.from_hparams(base_hparams)

print("Base editor loaded")

2026-05-03 23:25:33,837 - easyeditor.editors.editor - INFO - Instantiating model
2026-05-03 23:25:33,837 - easyeditor.editors.editor - INFO - Instantiating model
05/03/2026 23:25:33 - INFO - easyeditor.editors.editor -   Instantiating model
05/03/2026 23:25:33 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:33 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
05/03/2026 23:25:34 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:34 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
05/03/2026 23:25:34 - INFO - httpx -   HTTP Request: HEAD htt

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

05/03/2026 23:25:37 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
05/03/2026 23:25:38 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:38 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
05/03/2026 23:25:38 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
05/03/2026 23:25:38 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:38 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json

Base editor loaded


In [17]:
import gc
import torch

# Clear old experiment objects before starting a new run
try:
    del editor
except NameError:
    pass

try:
    del edited_model
except NameError:
    pass

try:
    del easy_case_df, robust_df, robust_df_v2, summary_row
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [18]:
sweep_k_rows = []

for k in [2,4,8]:
    try:
        del editor
    except NameError:
        pass
    
    try:
        del edited_model
    except NameError:
        pass
    
    try:
        del easy_case_df, robust_df, robust_df_v2, summary_row
    except NameError:
        pass
    
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    run_name = f"sweep_k_train1000_eval100_k{k}"

    summary_row, easy_case_df, robust_df, robust_df_v2, editor = run_one_ike_experiment(
        train_size=1000,
        eval_size=100,
        k=k,
        run_name=run_name,
        editor=None,
        max_new_tokens=32,
    )

    sweep_k_rows.append(summary_row)

sweep_k_table = pd.DataFrame(sweep_k_rows)

sweep_k_table.to_csv(OUT_DIR / "sweep_k_summary.csv", index=False)

sweep_k_table[
    [
        "train_size",
        "eval_size",
        "k",
        "ike_pre_rewrite",
        "ike_post_rewrite",
        "ike_post_rephrase",
        "ike_post_locality",
        "ike_post_portability",
        "robust_v2_rewrite",
        "robust_v2_rephrase",
        "robust_v2_locality",
        "robust_v2_portability",
        "elapsed_sec",
    ]
]

05/03/2026 23:25:40 - INFO - sentence_transformers.SentenceTransformer -   Use pytorch device_name: cuda:0
05/03/2026 23:25:40 - INFO - sentence_transformers.SentenceTransformer -   Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
05/03/2026 23:25:40 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:40 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"



RUN: sweep_k_train1000_eval100_k2
TRAIN_SIZE=1000, EVAL_SIZE=100, k=2
Loading train_ds size=1000


05/03/2026 23:25:41 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:41 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
05/03/2026 23:25:41 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:41 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
05/03/2026 23:25:41 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/re

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

05/03/2026 23:25:41 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:41 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
05/03/2026 23:25:41 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05/03/2026 23:25:41 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
05/03/2026 23:25:41 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=f

...

First eval prompt: When was the inception of IAAF Combined Events Challenge?
First target_new: 2006
First portability answer: Athletics


  0%|          | 0/100 [00:00<?, ?it/s]05/04/2026 00:30:59 - INFO - sentence_transformers.SentenceTransformer -   Use pytorch device_name: cuda:0
05/04/2026 00:30:59 - INFO - sentence_transformers.SentenceTransformer -   Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
05/04/2026 00:31:00 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
05/04/2026 00:31:00 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
05/04/2026 00:31:00 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
05/04/2026 00:31:00 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

05/04/2026 00:31:01 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
05/04/2026 00:31:01 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
05/04/2026 00:31:01 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
05/04/2026 00:31:01 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
05/04/2026 00:31:01 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=f

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 3.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.41 GiB is allocated by PyTorch, and 22.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import gc
import torch

sweep_train_rows = []

for train_size in [1000, 2000, 5000]:
    # Clear old experiment objects before starting a new run
    try:
        del editor
    except NameError:
        pass
    
    try:
        del edited_model
    except NameError:
        pass
    
    try:
        del easy_case_df, robust_df, robust_df_v2, summary_row
    except NameError:
        pass
    
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    run_name = f"sweep_train_train{train_size}_eval100_k4"

    summary_row, easy_case_df, robust_df, robust_df_v2, editor = run_one_ike_experiment(
        train_size=train_size,
        eval_size=100,
        k=4,
        run_name=run_name,
        editor=None,
        max_new_tokens=32,
    )

    sweep_train_rows.append(summary_row)

sweep_train_table = pd.DataFrame(sweep_train_rows)

sweep_train_table.to_csv(OUT_DIR / "sweep_train_size_summary.csv", index=False)

sweep_train_table[
    [
        "train_size",
        "eval_size",
        "k",
        "ike_pre_rewrite",
        "ike_post_rewrite",
        "ike_post_rephrase",
        "ike_post_locality",
        "ike_post_portability",
        "robust_v2_rewrite",
        "robust_v2_rephrase",
        "robust_v2_locality",
        "robust_v2_portability",
        "elapsed_sec",
    ]
]